In [1]:
print(123)

123


In [39]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")  # Load the ground truth file from the previous notebook

#  We'll use the ground truth CSV from the previous lesson and set up the search index here.

In [40]:
# For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.

In [41]:
df_ground_truth

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf
...,...,...
390,Why do I get a 401 Client Error when using the...,4b30b918bc
391,What's the easiest way to force-install reques...,4b30b918bc
392,Can I install requests straight from the GitHu...,4b30b918bc
393,"If pip keeps pulling requests v2.28, what exac...",4b30b918bc


In [42]:
df_ground_truth.head() # see top 5

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf


In [43]:
ground_truth = df_ground_truth.to_dict(orient="records") #turn df result into dictionary form

In [44]:
# Use the same ingest.py file we downloaded in the previous notebook.
# Load the documents and build a minsearch index

In [45]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [46]:
index.search

<bound method Index.search of <minsearch.minsearch.Index object at 0x12cd648b0>>

In [47]:
index.search("What is the course about?")

[{'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson

In [48]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson

In [49]:
# Wrap the search call in a function called text_search. The name is deliberate. 
# Later we'll write vector_search or a hybrid version and run the exact same evaluation on it. 
# Everything downstream only needs a function that takes a query and returns results, so we can swap one for another. 
# That mirrors how RAG works: the retrieval step doesn't care which search function sits behind it.

In [50]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

# Collecting relevance data

In [51]:
# Start with one ground truth record
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [52]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [53]:
# Run search for this question

In [54]:
results = text_search(q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.'},
 {'id': '610ccb23c0',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'What happens to code s

In [55]:
# First, compare the retrieved document IDs with the correct document ID

In [56]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
5cc511f85b == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


In [57]:
# Then turn this comparison into a relevance list. 
# In this lesson, relevance means whether a retrieved document is the correct document for this question.
# This gives a list of 0 and 1 values. 1 means the retrieved document has the same ID as the correct document

In [58]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [59]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [60]:
# For the first ground truth record, the relevance list is:

In [61]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [62]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 1, 0, 0]

Where can I watch the live sessions for the course, and how do I ask questions during them?


[1, 0, 0, 0, 0]

In [63]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 0, 0, 0]

Why does WSL2 say the model needs more memory even though my PC has enough RAM?


[1, 0, 0, 0, 0]

In [64]:
# Now do the same thing for all ground truth questions:

In [65]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [66]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/395 [00:00<?, ?it/s]

In [67]:
relevance

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0,

In [68]:
# Call it for the first 15 ground truth questions:

ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [69]:
relevance_total_text


[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [70]:
# Each entry in relevance_total_text is a relevance list. This is enough to check that the function works before we run it for the full dataset.

In [71]:
# Next, make the relevance functions generic. 
# We start with text search, but later we may want to evaluate vector search, hybrid search, or another retrieval method. 
# The relevance logic is the same. Only the search function changes.

In [72]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [73]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [74]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

In [75]:
sample = relevance_total[:15]

In [76]:
sample

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [82]:
13 / 15 # Hit rate

0.8666666666666667

In [79]:
# Since this rate isnt realistic, will replace the groud truthe file with one, with more realistic questions but had same results so will proceed for thi spractice

In [81]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt

13

In [84]:
cnt / len(sample) # Hit rate

0.8666666666666667

In [85]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [86]:
hit_rate(relevance)

0.6354430379746835

In [91]:
# Calculate MRR

total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.6777777777777777

In [92]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [93]:
mrr(sample)

0.6777777777777777

In [94]:
mrr(relevance)

0.5531645569620254

In [95]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [96]:
evaluate(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6354430379746835, 'mrr': 0.5531645569620254}

# Search Parameter Tuning

In [ ]:
# Trying different boosts
# Start with a search function where the question boost is configurable

In [97]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


In [98]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.640506329113924, 'mrr': 0.559071729957806}

In [100]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [101]:
# Grid search 
# Evaluate several boost values

for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.6683544303797468, 'mrr': 0.5728691983122364}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.6734177215189874, 'mrr': 0.5763291139240507}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.6354430379746835, 'mrr': 0.5531645569620254}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.6227848101265823, 'mrr': 0.5363291139240507}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.6075949367088608, 'mrr': 0.5140506329113924}


In [102]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [103]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [104]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
18,2.0,4.0,0.1,0.734177,0.632785
34,5.0,10.0,0.2,0.734177,0.632574
19,2.0,4.0,0.2,0.734177,0.632278
35,5.0,10.0,0.5,0.734177,0.632278
3,1.0,2.0,0.1,0.734177,0.632278
33,5.0,10.0,0.1,0.736709,0.631688
4,1.0,2.0,0.2,0.734177,0.630633
20,2.0,4.0,0.5,0.731646,0.626624
7,1.0,4.0,0.2,0.729114,0.618608
6,1.0,4.0,0.1,0.726582,0.618523
